<a href="https://colab.research.google.com/github/srikarspace/Hands-On-Large-Language-Models/blob/main/Colab_Notebooks/Chapter_1_Introduction_to_Language_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 1 - Introduction to Language Models

**Language AI:** Teaching computers how to work with human language, read it, understand it and write it. Computers only understand only numbers and the whole 70 year story was how to turn words into numbers so that they can keep their meaning.

---

## Story in 6 Steps

### 1. Bag of words - Count words
- Lets count words in a sentance and derive meaning from them. Ex: I Love pizza has three words, we put it in a bag and we can count, derive meaning from love and pizza.
- Its like understand a book by most frequent words in it.
- But order and meaning are lost.
  - Dog bites man vs Man bites dog. Both have same count but have different meaning because of order.
  - Bank has two meanings money and river. How to choose?

### 2. Words to embeddings(word2vector) - words as points in a map
- Each word gets a list of numbers(vector) in a giant map.
- Words with similar meaning sit together. Cat near dog far from car. Ex: King - Man + Women ~ Queen.
- It learns meaning by reding lot of text and keeps similar words together.
- But one fixed location per word. So above Bank is still stuck with one meaning.

### 3. Attention - Focus on what matters
- Older models(RNNs) used to keep read word-by-word and squeze the meaning into a small summary but if a long text was provided it would forget start. It would share meaning from one word to another.
- **Attention** looks at previous words to understand which word matters the most and use them to predict the current word.
  - Ex: "The animal didn't cross the street because it was tired", attention links "it" to "animal".

### 4. Transformers - Attention is all you need
- Drop word-by-word reading. Use only attention and read all words at once(parallel). Fater to train a model.
- This has two parts. **Encoding**(reads text) and **Decoding**(writes text)
- Word meaning now depends on context. **Contextualized Embeddings**
  - Ex: Banks gets meaning/numbers next to river or money based on before context.

### 5. Two families for Transformers
|| Bert Style | GPT Style |
|---|---|----|
|Uses|Encoder only|Decoder only|
|Skill|Understanding (Representing models)|Writing(Generating models)|
|Analogy|Great reader|Great autocomplete|
|Good For|Classification, search, clustering|Writing, code, Chatbot|
|How trained|Fill-in-the-blank ("I love [MASK]")|Pridict the next word|

- BERT (2018) = reader. GPT (2018+) = writer. Both are "LLMs" in the book's broad sense.

### 6. Generative Era
- Scale up GPT style models. Trained on more data. Tuned for chat. Ex: ChatGPT
- Closed source vs Open source Models

---

## What is an "LLM" really?
- **Large** = number of parameters. Billions of text. Now moving towards Trillions.
- **Language Models** = Predicts what comes next.
  - Ex: The sky is ___? model gives "Blue" high and "Black" low.
- Generative text meaning predict a word, add it, predict again and repeat.

---

## How LLMs are Trained - 2 Steps
1. **Pretraining** = read a huge pile of internet text, learn to predict next word. Very expensive.
  - Result = **Base/Foundational Model**. Knows language but not how to follow instructions.
2. **Fine Tuning** = Train bit more on following good behaviour. Ex: Follow instructions, answer questions. Cheap compared to step 1.
  - Result = **Chat Model**

**Analogy**: Pretraining = going to school for 12 years (general knowledge). Fine-tuning = job training for one role.

---

## Limitations and Responsibilities
- Hallucination: sounds confidat but makes things up
- Bias: Its trained on human generated text so it would behaive like human.
- Transparency: We don't know what data it got trained on.
- Missuse, harm, copyright and data issues.
- Enery usage is high.
- We need to verify for important information.




### [OPTIONAL] - Installing Packages

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---

In [ ]:
%%capture
!pip install transformers==4.41.2 accelerate==0.31.0

# Phi-3

The first step is to load our model onto the GPU for faster inference. Note that we load the model and tokenizer separately (although that isn't always necessary).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer - why seprately? they both do different jobs

# model does the thinking/processes tokens
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda", # put it on a GPU
    torch_dtype="auto", # pick number precision
    trust_remote_code=False,
)
# text <-> numbers/tokens
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Although we can now use the model and tokenizer directly, it's much easier to wrap it in a `pipeline` object:

In [ ]:
from transformers import pipeline

# Create a pipeline - it helps with tokenize, generate and decode for us
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False, # return only the answer, not the prompt
    max_new_tokens=500, # Max tokens it should generate. Stop after 500
    do_sample=False # always pick most likely next token
)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Finally, we create our prompt as a user and give it to the model:

In [ ]:
# The prompt (user input / query)
messages = [
    {"role": "user", "content": "Create a funny joke about dogs."}
]

# Generate output
output = generator(messages)
print(output[0]["generated_text"])

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Why don't dogs make good secret agents? Because they always bark at the wrong time!


## 30-second recap
- Language AI = turning words into numbers that keep meaning.
- Bag of words -> word2vec embeddings -> attention -> transformer -> BERT (reader) / GPT (writer) -> chat LLMs.
- LLM = giant next-word predictor, pretrained on the internet, then fine-tuned to be helpful.
- Beware hallucination and bias.

---

## Self-quiz (answers at bottom)
1. Why was bag of words a poor way to represent language?
2. What does a word embedding capture that bag of words doesn't?
3. What problem did attention solve for older word-by-word models?
4. Encoder-only vs decoder-only: which understands, which writes?
5. What's the difference between a base model and an instruct model?
6. In the notebook, what does `do_sample=False` do?
7. Why are the model and tokenizer loaded separately?

<details><summary>Answers</summary>

1. Ignores word order and context; "bank" always the same.
2. Meaning: similar words get nearby vectors.
3. Forgetting early words in long sentences / one-summary bottleneck; attention lets each word look at all others.
4. Encoder (BERT) understands; decoder (GPT) writes.
5. Base = pretrained next-word predictor; instruct = further fine-tuned to follow instructions.
6. Greedy decoding: always the most likely token, so output is deterministic.
7. They do different jobs: tokenizer converts text <-> tokens, model processes tokens.
</details>
